In [2]:
from models import *
from config import Config
from test_quant import str2model, build_transform
import torchvision.datasets as datasets
import torch

cfg = Config(True, True, 'minmax')
model_name = 'deit_tiny'
quantized_file = model_name + "_quantized.pth"

In [3]:


model_type = model_name.split('_')[0]
if model_type == 'deit':
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    crop_pct = 0.875
elif model_type == 'vit':
    mean = (0.5, 0.5, 0.5)
    std = (0.5, 0.5, 0.5)
    crop_pct = 0.9
elif model_type == 'swin':
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    crop_pct = 0.9
else:
    raise NotImplementedError


train_transform = build_transform(mean=mean, std=std, crop_pct=crop_pct)
val_transform = build_transform(mean=mean, std=std, crop_pct=crop_pct)

traindir = 'imagenet/train'
valdir = 'imagenet/val'

val_dataset = datasets.ImageFolder(valdir, val_transform)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

# train_dataset = datasets.ImageFolder(traindir, train_transform)
# train_loader = torch.utils.data.DataLoader(
#     train_dataset,
#     batch_size=100,
#     shuffle=True,
#     num_workers=8,
#     pin_memory=True,
#     drop_last=True,
# )

In [4]:
from test_quant import accuracy, AverageMeter

# top1 = AverageMeter()
# top5 = AverageMeter()

bit_config = [4]*50
for image, target in val_loader:
    target = target.to('cuda')
    break
    # output, FLOPs, distance = model(image, bit_config, False)
    # prec1, prec5 = accuracy(output.data, target, topk=(1, 5))
    # top1.update(prec1)
    # top5.update(prec5)

In [ ]:
outputs = {}

def get_activation(name):
    def hook(model, input, output):
        if isinstance(output, torch.Tensor):
            output = output.detach()
        elif isinstance(output, (list, tuple)):
            # Detach each tensor in the tuple/list
            output= [o.detach() if torch.is_tensor(o) else o for o in output]

        if isinstance(input, torch.Tensor):
            input = input.detach()
        elif isinstance(output, (list, tuple)):
            # Detach each tensor in the tuple/list
            input= [o.detach() if torch.is_tensor(o) else o for o in input]
        
        outputs[name] = {'layer':model, 'output':output, 'input':input.detach()}
    return hook

model = torch.load(quantized_file, map_location=torch.device('cuda'), weights_only=False)
model.eval()

# Register a hook for every submodule
for name, layer in model.named_modules():
    layer.register_forward_hook(get_activation(name))

# Example input
x = image[0].unsqueeze(0)
_ = model(x, bit_config, False)

# Now outputs[name] contains the output of that layer


AttributeError: 'tuple' object has no attribute 'detach'

In [6]:
for key in outputs.keys():
    print(key)

qact_input.quantizer
qact_input
patch_embed.proj.quantizer
patch_embed.proj
patch_embed.qact_before_norm
patch_embed.norm
patch_embed.qact.quantizer
patch_embed.qact
patch_embed
qact_embed.quantizer
qact_embed
qact_pos.quantizer
qact_pos
qact1.quantizer
qact1
pos_drop
blocks.0.norm1
blocks.0.attn.qact0.quantizer
blocks.0.attn.qact0
blocks.0.attn.qkv.quantizer
blocks.0.attn.qkv
blocks.0.attn.qact1.quantizer
blocks.0.attn.qact1
blocks.0.attn.qact_attn1.quantizer
blocks.0.attn.qact_attn1
blocks.0.attn.log_int_softmax
blocks.0.attn.attn_drop
blocks.0.attn.qact2.quantizer
blocks.0.attn.qact2
blocks.0.attn.proj.quantizer
blocks.0.attn.proj
blocks.0.attn.qact3.quantizer
blocks.0.attn.qact3
blocks.0.attn.proj_drop
blocks.0.attn
blocks.0.drop_path
blocks.0.qact2.quantizer
blocks.0.qact2
blocks.0.norm2
blocks.0.mlp.qact0.quantizer
blocks.0.mlp.qact0
blocks.0.mlp.fc1.quantizer
blocks.0.mlp.fc1
blocks.0.mlp.act
blocks.0.mlp.qact1.quantizer
blocks.0.mlp.qact1
blocks.0.mlp.drop
blocks.0.mlp.fc2.quan

In [7]:
model.blocks[0].attn.qkv

QLinear(
  in_features=192, out_features=576, bias=True
  (quantizer): UniformQuantizer()
)

In [117]:
qkv_in = outputs['blocks.0.norm1']['output']
qkv_in_scale = outputs['blocks.0.attn.qact0']['layer'].quantizer.scale

quant_act = qkv_in / qkv_in_scale

In [90]:
qkv_layer = outputs['blocks.0.attn.qkv']['layer']
range_shape = qkv_layer.quantizer.get_reshape_range(qkv_in)

weights = qkv_layer.weight
bias = qkv_layer.bias.unsqueeze(-1)

weight_scale = qkv_layer.quantizer.dic_scale[qkv_layer.quantizer.bit_type.name]
weight_zp = qkv_layer.quantizer.dic_zero_point[qkv_layer.quantizer.bit_type.name]

weight_scale = weight_scale.reshape(range_shape)
weight_zp = weight_zp.reshape(range_shape)

In [96]:
quant_weights = torch.round(weights / weight_scale).clamp(qkv_layer.quantizer.bit_type.lower_bound, qkv_layer.quantizer.bit_type.upper_bound)
quant_bias = torch.round(bias / weight_scale).clamp(qkv_layer.quantizer.bit_type.lower_bound, qkv_layer.quantizer.bit_type.upper_bound).squeeze(-1)

In [123]:
torch.nn.functional.linear(outputs['blocks.0.attn.qact0']['layer'].quantizer(qkv_in), qkv_layer.quantizer(weights), bias.squeeze(-1)) #* qkv_in_scale * qkv_layer.quantizer.dic_scale[qkv_layer.quantizer.bit_type.name] 

tensor([[[ 0.0602,  0.0889,  1.2973,  ...,  0.2189,  0.0717,  0.0172],
         [ 0.0153,  0.0557,  0.7348,  ...,  1.6212,  2.0717,  2.4274],
         [-0.0482,  0.2808,  0.6323,  ..., -2.2050,  0.1537,  0.6832],
         ...,
         [ 0.2853, -0.3242,  1.6762,  ...,  1.4337,  0.0111, -0.0453],
         [ 0.3127, -0.2627,  1.2348,  ...,  2.3282, -1.3502, -0.6117],
         [ 0.1276, -0.1758,  1.5805,  ...,  0.1173, -0.4303,  0.4684]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [61]:
outputs['blocks.0.attn.qkv']['output']
# qkv_out / (qkv_in_scale * qkv_layer.quantizer.dic_scale[qkv_layer.quantizer.bit_type.name])

tensor([[[-0.0921,  0.1396,  1.4936,  ...,  0.0841,  0.1439, -0.0101],
         [-0.0447, -0.0322,  0.6323,  ...,  1.5450,  2.2318,  2.0543],
         [-0.1688,  0.2910,  0.3276,  ..., -1.8846, -0.7096,  0.5739],
         ...,
         [ 0.3845, -0.1196,  1.4829,  ...,  1.2306,  0.0678,  0.2067],
         [ 0.3024, -0.1226,  1.3754,  ...,  1.6720, -1.0924, -0.3949],
         [ 0.2800,  0.0898,  1.5171,  ...,  0.0548, -0.4127,  0.3785]]],
       device='cuda:0')

In [12]:
quant_scale = outputs['blocks.0.attn.qact0.quantizer']['layer'].scale

In [13]:
outputs

{'qact_input.quantizer': {'layer': UniformQuantizer(),
  'output': tensor([[[[-0.5625, -0.3750, -0.5312,  ...,  1.0000,  0.2812,  0.6875],
            [-0.6562, -0.8125, -0.5000,  ...,  0.8125,  0.5938,  0.0938],
            [-0.7188, -0.1875,  0.0625,  ...,  0.7500,  0.9688,  0.1250],
            ...,
            [-0.3750, -0.4062, -0.3750,  ..., -0.0938, -0.3750, -0.5625],
            [-0.3125, -0.3125, -0.3750,  ..., -0.5938,  0.1875,  0.7812],
            [-0.2500, -0.2500, -0.1562,  ..., -0.2500, -0.3750, -0.3438]],
  
           [[-0.7188, -0.4688, -0.0312,  ...,  1.3438,  0.6562,  1.0938],
            [-0.5625, -0.7500, -0.4062,  ...,  1.1562,  0.8750,  0.4375],
            [-0.5938, -0.1562,  0.0625,  ...,  1.0625,  1.3750,  0.5625],
            ...,
            [-0.2500, -0.3125, -0.3125,  ..., -0.0312, -0.1875, -0.1562],
            [ 0.0312, -0.1562, -0.2812,  ..., -0.1875,  0.4688,  1.2500],
            [ 0.7812,  0.3750,  0.3750,  ...,  0.1562, -0.0938, -0.2188]],
  
     

In [14]:
quant_scale

tensor([0.0156], device='cuda:0')